# GeoJSON to GeoPackage (GPKG) Converter

## Purpose

Convert GeoJSON input into a GeoPackage (`.gpkg`) file so the geometries can be used by
downstream spatial tooling (Sedona, GeoTools, QGIS, the OPAS context extractor, etc.) that
expects GPKG rather than raw GeoJSON.

## What this notebook does

1. **Read** the source GeoJSON — from a DBFS / mounted path (e.g. `/mnt/opas/...`) or a
   catalog table holding GeoJSON / WKT geometry.
2. **Parse / build geometry** — load the features into a (Geo)DataFrame, validating the CRS
   (defaults to `EPSG:4326` / WGS84) and the geometry column.
3. **Write** the features out as a single `.gpkg` file (one layer per feature collection),
   then place it at the target output path.

## Inputs & outputs

- **Input:** GeoJSON file(s) or a table with a geometry/WKT column.
- **Output:** a `.gpkg` GeoPackage written to the configured output directory.
- **CRS:** `EPSG:4326` unless overridden.

## Notes

- Geometry is the unit that carries across — confirm the source CRS before writing, as GPKG
  stores the CRS in the layer metadata.
- For large inputs, prefer GeoPandas/Fiona locally (`.toPandas()` first) or a Sedona-based
  write path; pick whichever matches the data volume.


In [2]:
# Install the geospatial packages needed to read GeoJSON and write GeoPackage (GPKG).
# Use pyogrio (NOT fiona): pyogrio ships manylinux wheels with GDAL bundled, so it needs no
# system gdal-config. fiona would otherwise try to build from source and fail on the cluster
# with "A GDAL API version must be specified".
# %pip installs into the notebook's Python env; the kernel restarts automatically afterwards.
%pip install geopandas pyogrio shapely pyproj


  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached shapely-2.1.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 1.6 MB/s  0:00:15m0:00:0100:01
Using cached shapely-2.1.2-cp314-cp314-macosx_11_0_arm64.whl (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 2.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [pyogrio]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Read the source GeoJSON into a GeoDataFrame.
# On Databricks, GeoPandas reads from the local FUSE mount, so prefix DBFS paths with /dbfs.

import geopandas as gpd

# >>> Set the input GeoJSON path here <<<
input_geojson_path = "/Users/sawan.darekar/Desktop/workspace_data/SEACO-6182-ESP-Source-Validation/ZetOwN33zr_oKJ2YMu5Z1_out_prepped_catastro_0aeaa18_part01.geojson"

gdf = gpd.read_file(input_geojson_path)

# Default to WGS84 when the source carries no CRS
if gdf.crs is None:
    gdf = gdf.set_crs("EPSG:4326")

print(f"Read {len(gdf)} features from: {input_geojson_path}")
print(f"CRS: {gdf.crs}")
print(f"Geometry types: {gdf.geom_type.value_counts().to_dict()}")
print(f"Columns: {list(gdf.columns)}")

gdf.head()


Read 10 features from: input.geojson
CRS: EPSG:4326
Geometry types: {'Point': 10}
Columns: ['id', 'name', 'category', 'geometry']


,id,name,category,geometry
0,1,Madrid,city,POINT (-3.7038 40.4168)
1,2,Barcelona,city,POINT (2.1734 41.3851)
2,3,Valencia,city,POINT (-0.3763 39.4699)
3,4,Sevilla,city,POINT (-5.9845 37.3891)
4,5,Zaragoza,city,POINT (-0.8891 41.6488)


In [ ]:
# Convert the GeoDataFrame to a GeoPackage (.gpkg) and place it at the target DBFS path.
# GeoPandas/Fiona write to the local FUSE mount, so build the .gpkg under /dbfs and use the
# GPKG driver; layer_name becomes the layer inside the GeoPackage.

import os

# >>> Set the output GPKG path and layer name here <<<
output_gpkg_path = "/Users/sawan.darekar/Desktop/workspace_data/SEACO-6182-ESP-Source-Validation/ZetOwN33zr_oKJ2YMu5Z1.gpkg"
layer_name = "features"

# Overwrite any existing file so re-runs are clean
if os.path.exists(output_gpkg_path):
    os.remove(output_gpkg_path)

os.makedirs(os.path.dirname(output_gpkg_path), exist_ok=True)

gdf.to_file(output_gpkg_path, layer=layer_name, driver="GPKG")

print(f"Wrote {len(gdf)} features to: {output_gpkg_path} (layer: {layer_name})")


Wrote 10 features to: /Users/sawan.darekar/Desktop/tomtom_git_workspace/databricks-workspace/output.gpkg (layer: features)
